<a href="https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1 — The Anatomy of Growing Content

The paper reports that content trending upward tends to be longer, younger, and slightly better positioned than content trending downward. For example, growing pages averaged 3.2K words and 184 days old, compared with 2.3K words and 230 days for declining pages. The paper labels this finding as "CONFIRMED". :contentReference[oaicite:0]{index=0}

**My methodology question: where does the label come from?**

The growing/declining groups are defined using trend direction, which is calculated from the change in impressions between the most recent 30 days and the previous 30 days. A page is classified as "Up" if impressions increase by more than 10% and "Down" if they decline by more than 10%. :contentReference[oaicite:1]{index=1}

My question would therefore be whether this label is measuring the underlying outcome we are interested in, or simply describing short-term movement in impressions. A page being classified as "growing" does not necessarily mean that its content quality improved, and a "declining" page does not necessarily mean that its content became worse.

I would not treat this as a flaw in the analysis, because the paper is clear that it is making an observational comparison. However, I would interpret the finding as a measured association between the trend label and page characteristics rather than evidence that being younger or longer causes growth.


### Finding 6 — AI Traffic: A Different Signal

The paper finds that pages receiving more AI referral traffic have substantially more impressions but a weaker average Google position than pages receiving no AI traffic. The paper concludes that AI-referral visibility appears to behave differently from traditional organic search visibility. :contentReference[oaicite:2]{index=2}

**My methodology question: does the validation design carry the claim?**

The evidence is based on comparisons between AI-traffic buckets within the portfolio, rather than a predictive or experimental validation design. The paper itself describes its main evidence as direct aggregate comparisons and says that the ML analysis is exploratory. :contentReference[oaicite:3]{index=3}

My question would therefore be whether the observed differences are robust across different clients, time periods, and page types, rather than being driven by the particular portfolio or snapshot used in the study. This matters because the paper is using a local cached snapshot and acknowledges that the study is observational. :contentReference[oaicite:4]{index=4}

I think the paper is appropriately cautious here: it describes the conclusion as a narrower observation that AI-referral visibility is real and behaves differently, rather than claiming that AI traffic causes better or worse Google performance. I would therefore treat this finding as directional evidence that supports further investigation, rather than as a universal SEO rule.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

march_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,

        BOOL_OR(gsc_data_available) AS gsc_data_available,
        BOOL_OR(ga4_data_available) AS ga4_data_available

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

import numpy as np

march_df["gsc_ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

march_df["engagement_rate"] = (
    march_df["ga4_engaged_sessions"] /
    march_df["ga4_sessions"].replace(0, np.nan)
).fillna(0)

features = [
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]

X = march_df[features].copy()

# Replace infinite values
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values
X = X.fillna(0)

# 1. Prepare March dataset

proxy_df = march_df.copy()

print("Rows:", len(proxy_df))

feature_cols = [
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]


import numpy as np

proxy_df["log_impressions"] = np.log1p(
    proxy_df["gsc_impressions"]
)

proxy_df["log_sessions"] = np.log1p(
    proxy_df["ga4_sessions"]
)

proxy_df["log_engagement"] = np.log1p(
    proxy_df["engagement_rate"]
)

model_features = [
    "log_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "log_sessions",
    "log_engagement"
]

print("Model features:", model_features)

# 3. Create March-only proxy target

proxy_df["impression_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_impressions"]
    .rank(pct=True)
)

proxy_df["ctr_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_ctr"]
    .rank(pct=True)
)

proxy_df["position_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_avg_position"]
    .rank(pct=True)
)

proxy_df["engagement_percentile"] = (
    proxy_df.groupby("client_hash_id")["engagement_rate"]
    .rank(pct=True)
)

# A page is "review-worthy" if:
# - it has relatively high visibility within its client
# - AND it has at least one sign of underperformance

proxy_df["target"] = (
    (proxy_df["impression_percentile"] >= 0.75)
    &
    (
        (proxy_df["ctr_percentile"] <= 0.25)
        |
        (proxy_df["position_percentile"] >= 0.75)
        |
        (proxy_df["engagement_percentile"] <= 0.25)
    )
).astype(int)


# 4. Define X, y and groups

X = proxy_df[model_features].copy()

# Missing numeric values are filled with 0.
X = X.fillna(0)

y = proxy_df["target"]

groups = proxy_df["client_hash_id"]


# 5. Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# 6. Train Random Forest classifier

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


# 7. Generate ML scores

test_results = proxy_df.iloc[test_idx].copy()

test_results["ml_score"] = rf.predict_proba(
    X_test
)[:, 1]

# 8. Rank pages using ML score

test_results = test_results.sort_values(
    "ml_score",
    ascending=False
).reset_index(drop=True)

test_results["ml_rank"] = (
    np.arange(len(test_results)) + 1
)

ml_top20 = test_results.head(20)







FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331437
Model features: ['log_impressions', 'gsc_ctr', 'gsc_avg_position', 'log_sessions', 'log_engagement']


In [3]:
# Recreate the exact Week-5 March-only proxy target

march_df["impression_percentile"] = (
    march_df.groupby("client_hash_id")["gsc_impressions"]
    .rank(pct=True)
)

march_df["ctr_percentile"] = (
    march_df.groupby("client_hash_id")["gsc_ctr"]
    .rank(pct=True)
)

march_df["position_percentile"] = (
    march_df.groupby("client_hash_id")["gsc_avg_position"]
    .rank(pct=True)
)

march_df["engagement_percentile"] = (
    march_df.groupby("client_hash_id")["engagement_rate"]
    .rank(pct=True)
)

march_df["target"] = (
    (march_df["impression_percentile"] >= 0.75)
    &
    (
        (march_df["ctr_percentile"] <= 0.25)
        |
        (march_df["position_percentile"] >= 0.75)
        |
        (march_df["engagement_percentile"] <= 0.25)
    )
).astype(int)

print("Target distribution:")
print(march_df["target"].value_counts())

print("\nTarget rate:")
print(march_df["target"].mean())

Target distribution:
target
0    319373
1     12064
Name: count, dtype: int64

Target rate:
0.03639907433388547


In [5]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

march_df["log_impressions"] = np.log1p(
    march_df["gsc_impressions"]
)

march_df["log_sessions"] = np.log1p(
    march_df["ga4_sessions"]
)

march_df["log_engagement"] = np.log1p(
    march_df["engagement_rate"]
)

features = [
    "log_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "log_sessions",
    "log_engagement"
]

X = march_df[features]
y = march_df["target"]
groups = march_df["client_hash_id"]


def make_model():
    return RandomForestClassifier(
        class_weight="balanced",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=42
    )


# -------------------------
# BEFORE: row-level random split
# -------------------------

X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_before = make_model()
model_before.fit(X_train_before, y_train_before)

pred_before = model_before.predict_proba(X_test_before)[:, 1]

roc_before = roc_auc_score(y_test_before, pred_before)
ap_before = average_precision_score(y_test_before, pred_before)


# -------------------------
# AFTER: grouped-by-client split
# -------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_after = X.iloc[train_idx]
X_test_after = X.iloc[test_idx]

y_train_after = y.iloc[train_idx]
y_test_after = y.iloc[test_idx]

groups_train_after = groups.iloc[train_idx]
groups_test_after = groups.iloc[test_idx]

model_after = make_model()
model_after.fit(X_train_after, y_train_after)

pred_after = model_after.predict_proba(X_test_after)[:, 1]

roc_after = roc_auc_score(y_test_after, pred_after)
ap_after = average_precision_score(y_test_after, pred_after)


# -------------------------
# Comparison
# -------------------------

comparison = pd.DataFrame({
    "split": [
        "Before: row-level random",
        "After: grouped by client"
    ],
    "train_rows": [
        len(X_train_before),
        len(X_train_after)
    ],
    "test_rows": [
        len(X_test_before),
        len(X_test_after)
    ],
    "train_clients": [
        march_df.loc[X_train_before.index, "client_hash_id"].nunique(),
        groups_train_after.nunique()
    ],
    "test_clients": [
        march_df.loc[X_test_before.index, "client_hash_id"].nunique(),
        groups_test_after.nunique()
    ],
    "roc_auc": [
        roc_before,
        roc_after
    ],
    "average_precision": [
        ap_before,
        ap_after
    ]
})

comparison

,split,train_rows,test_rows,train_clients,test_clients,roc_auc,average_precision
0,Before: row-level random,265149,66288,55,54,0.966157,0.610797
1,After: grouped by client,300880,30557,44,11,0.880852,0.360497


### Interpretation

The row-level random split produced a ROC-AUC of 0.966 and an Average Precision of 0.610. Under the grouped-by-client split, these fell to 0.880 and 0.358 respectively.

This difference suggests that the easier row-level split may have benefited from similarities between pages from the same clients appearing in both the training and test sets. The grouped-by-client split is therefore a more conservative test of whether the model can rank pages for clients that were not seen during training.

I use the grouped-by-client result as the more appropriate validation result for this project. The model still shows useful ranking signal on the held-out clients, but the lower performance demonstrates that the stronger result from a random row-level split should not be presented as evidence of equivalent performance on unseen clients.

This validation is still limited to the March 2026 dataset and the March-only proxy target. It therefore measures performance against the proxy definition rather than demonstrating that the model predicts which pages will actually benefit from a content refresh.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final feature set used by the model
features = [
    "log_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "log_sessions",
    "log_engagement"
]

print("Final model features:")
print(features)

# Check for forbidden / target-derived columns
forbidden = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "trend_direction"
]

print("\nForbidden or target-related columns present in features:")
print([col for col in features if col in forbidden])

# Check train/test client overlap
train_clients = set(X_train.index.map(lambda x: march_df.loc[x, "client_hash_id"]))
test_clients = set(X_test.index.map(lambda x: march_df.loc[x, "client_hash_id"]))

print("\nClient overlap between train and test:")
print(len(train_clients.intersection(test_clients)))

Final model features:
['log_impressions', 'gsc_ctr', 'gsc_avg_position', 'log_sessions', 'log_engagement']

Forbidden or target-related columns present in features:
[]

Client overlap between train and test:
0


In [7]:
# Check which model features were also used to construct the target
target_source_variables = [
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]

overlap = []

for feature in features:
    if feature == "log_impressions":
        overlap.append(("log_impressions", "gsc_impressions"))
    elif feature == "gsc_ctr":
        overlap.append(("gsc_ctr", "gsc_ctr"))
    elif feature == "gsc_avg_position":
        overlap.append(("gsc_avg_position", "gsc_avg_position"))
    elif feature == "log_sessions":
        overlap.append(("log_sessions", "ga4_sessions"))
    elif feature == "log_engagement":
        overlap.append(("log_engagement", "engagement_rate"))

print("Features derived from variables used to construct target:")
for feature, target_variable in overlap:
    print(f"{feature} <- {target_variable}")

Features derived from variables used to construct target:
log_impressions <- gsc_impressions
gsc_ctr <- gsc_ctr
gsc_avg_position <- gsc_avg_position
log_sessions <- ga4_sessions
log_engagement <- engagement_rate


The final feature set contains five variables: log_impressions, gsc_ctr, gsc_avg_position, log_sessions and log_engagement. None of the forbidden product decision fields or trend_direction are included. The grouped train/test split also has zero client overlap.

There is no future-date leakage because the analysis is restricted to March 2026.

However, there is an important feature-target overlap. The proxy target was constructed using client-level percentiles of impressions, CTR, average position and engagement. These same underlying variables are then supplied to the model as features. Therefore, the model is learning from the same information that defines the target.

This makes the model evaluation partly circular: the ROC-AUC and average precision measure how well the Random Forest reproduces the March proxy definition, rather than how well it predicts a genuinely future outcome such as whether a page will benefit from a content refresh.

The grouped-by-client split addresses client overlap, but it does not remove this feature-target overlap. This is therefore the main leakage limitation of the current model.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim:

“The model can rank pages that should be prioritised for a content refresh.”

Safe rewrite:

“The model observed and measured differences between March 2026 pages using search visibility, position and engagement signals. Under the grouped-by-client evaluation, it provides a directional ranking for decision-support and review prioritisation. However, because the proxy target was constructed from the same signals used as model features, the results should not be interpreted as evidence that the model predicts which pages will benefit from a content refresh.”

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.